### PyTorch BiLSTM

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Load data
df = pd.read_csv("tox21.csv")
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]
df = df.dropna(subset=target_cols).reset_index(drop=True)
smiles = df['smiles'].values
labels = df[target_cols].astype(int).values

# Train-test split
X_train, X_temp, y_train, y_temp = train_test_split(smiles, labels, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Tokenizer
class CharTokenizer:
    def __init__(self, smiles_list):
        chars = sorted(set(''.join(smiles_list)))
        self.char2idx = {ch: i + 1 for i, ch in enumerate(chars)}
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx) + 1

    def encode(self, smiles, max_len=120):
        encoded = [self.char2idx.get(ch, 0) for ch in smiles[:max_len]]
        return encoded + [0] * (max_len - len(encoded))

tokenizer = CharTokenizer(X_train)
max_len = 120

class Tox21Dataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_len):
        self.inputs = [tokenizer.encode(smiles, max_len) for smiles in smiles_list]
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

train_loader = DataLoader(Tox21Dataset(X_train, y_train, tokenizer, max_len), batch_size=64, shuffle=True)
val_loader = DataLoader(Tox21Dataset(X_val, y_val, tokenizer, max_len), batch_size=64)
test_loader = DataLoader(Tox21Dataset(X_test, y_test, tokenizer, max_len), batch_size=64)

# Model
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        hidden = torch.cat((hidden[0], hidden[1]), dim=1)
        return self.fc(hidden)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMClassifier(tokenizer.vocab_size, embed_dim=128, hidden_dim=64, output_dim=12).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
def train(model, loader):
    model.train()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

# Evaluation
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x).cpu()
            y_pred.append(torch.sigmoid(logits))
            y_true.append(y)
    return torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()

for epoch in range(10):
    train(model, train_loader)

y_true, y_score = evaluate(model, test_loader)
y_pred = (y_score >= 0.5).astype(int)

# Metrics
metrics = {
    'Label': [],
    'AUC': [], 'Accuracy': [], 'F1': [], 'Precision': [], 'Recall': []
}

for i, label in enumerate(target_cols):
    try:
        auc = roc_auc_score(y_true[:, i], y_score[:, i])
    except:
        auc = np.nan
    metrics['Label'].append(label)
    metrics['AUC'].append(auc)
    metrics['Accuracy'].append(accuracy_score(y_true[:, i], y_pred[:, i]))
    metrics['F1'].append(f1_score(y_true[:, i], y_pred[:, i]))
    metrics['Precision'].append(precision_score(y_true[:, i], y_pred[:, i]))
    metrics['Recall'].append(recall_score(y_true[:, i], y_pred[:, i]))

df_metrics = pd.DataFrame(metrics)
df_metrics.loc['mean'] = df_metrics.mean(numeric_only=True)
df_metrics.loc['median'] = df_metrics.median(numeric_only=True)
print(df_metrics)

c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()

                Label       AUC  Accuracy   F1  Precision  Recall
0               NR-AR  0.580827  0.977492  0.0        0.0     0.0
1           NR-AR-LBD  0.850327  0.983923  0.0        0.0     0.0
2              NR-AhR  0.748517  0.948553  0.0        0.0     0.0
3        NR-Aromatase  0.726898  0.974277  0.0        0.0     0.0
4               NR-ER  0.457814  0.906752  0.0        0.0     0.0
5           NR-ER-LBD  0.447368  0.977492  0.0        0.0     0.0
6       NR-PPAR-gamma  0.854978  0.990354  0.0        0.0     0.0
7              SR-ARE  0.652668  0.938907  0.0        0.0     0.0
8            SR-ATAD5       NaN  1.000000  0.0        0.0     0.0
9              SR-HSE  0.271452  0.974277  0.0        0.0     0.0
10             SR-MMP  0.700337  0.954984  0.0        0.0     0.0
11             SR-p53  0.611111  0.983923  0.0        0.0     0.0
mean              NaN  0.627482  0.967578  0.0        0.0     0.0
median            NaN  0.640075  0.974277  0.0        0.0     0.0


In [2]:
# BiLSTM with pos_weight, validation, dropout, threshold tuning
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Load and clean dataset
df = pd.read_csv("tox21.csv")
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]
df = df.dropna(subset=target_cols).reset_index(drop=True)
smiles = df['smiles'].values
labels = df[target_cols].astype(int).values

# Split
token_max_len = 120
X_train, X_temp, y_train, y_temp = train_test_split(smiles, labels, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Char tokenizer
class CharTokenizer:
    def __init__(self, smiles_list):
        chars = sorted(set(''.join(smiles_list)))
        self.char2idx = {ch: i + 1 for i, ch in enumerate(chars)}
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx) + 1

    def encode(self, smiles, max_len=120):
        encoded = [self.char2idx.get(ch, 0) for ch in smiles[:max_len]]
        return encoded + [0] * (max_len - len(encoded))

tokenizer = CharTokenizer(X_train)

# Dataset class
class Tox21Dataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_len):
        self.inputs = [tokenizer.encode(smiles, max_len) for smiles in smiles_list]
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

# DataLoaders
train_loader = DataLoader(Tox21Dataset(X_train, y_train, tokenizer, token_max_len), batch_size=64, shuffle=True)
val_loader = DataLoader(Tox21Dataset(X_val, y_val, tokenizer, token_max_len), batch_size=64)
test_loader = DataLoader(Tox21Dataset(X_test, y_test, tokenizer, token_max_len), batch_size=64)

# Pos_weight calculation
pos_weight = torch.tensor((y_train == 0).sum(axis=0) / (y_train == 1).sum(axis=0), dtype=torch.float)

# BiLSTM model with Dropout (ADDED HERE)
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)  # <-- ADDED DROPOUT
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        hidden = torch.cat((hidden[0], hidden[1]), dim=1)
        x = self.dropout(hidden)  # <-- ADDED DROPOUT
        return self.fc(x)

# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMClassifier(tokenizer.vocab_size, 128, 64, 12).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))  # <-- ADDED pos_weight
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop with validation monitoring
best_val_auc = 0
for epoch in range(30):  # <-- INCREASED TO 30 EPOCHS
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    # Evaluate on validation set
    model.eval()
    y_val_true, y_val_score = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            logits = model(x).cpu()
            y_val_score.append(torch.sigmoid(logits))
            y_val_true.append(y)
    y_val_true = torch.cat(y_val_true).numpy()
    y_val_score = torch.cat(y_val_score).numpy()
    aucs = [roc_auc_score(y_val_true[:, i], y_val_score[:, i]) for i in range(12) if len(np.unique(y_val_true[:, i])) > 1]
    mean_auc = np.mean(aucs)
    print(f"Epoch {epoch + 1}, Validation AUC: {mean_auc:.4f}")

# Predict on test set
def predict(model, loader):
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x).cpu()
            y_score.append(torch.sigmoid(logits))
            y_true.append(y)
    return torch.cat(y_true).numpy(), torch.cat(y_score).numpy()

y_true, y_score = predict(model, test_loader)

# Tune threshold per class on val set
def tune_threshold(y_true, y_score):
    thresholds = []
    for i in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, 0
        for t in np.arange(0.1, 0.9, 0.05):
            pred = (y_score[:, i] >= t).astype(int)
            score = f1_score(y_true[:, i], pred)
            if score > best_f1:
                best_f1 = score
                best_t = t
        thresholds.append(best_t)
    return thresholds

# Use val set to tune thresholds
y_val_true, y_val_score = predict(model, val_loader)
optimal_thresholds = tune_threshold(y_val_true, y_val_score)

# Apply thresholds to test set
y_pred = np.zeros_like(y_score)
for i in range(12):
    y_pred[:, i] = (y_score[:, i] >= optimal_thresholds[i]).astype(int)

# Metrics
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]
metrics = {'Label': [], 'AUC': [], 'Accuracy': [], 'F1': [], 'Precision': [], 'Recall': []}
for i, label in enumerate(target_cols):
    try:
        auc = roc_auc_score(y_true[:, i], y_score[:, i])
    except:
        auc = np.nan
    metrics['Label'].append(label)
    metrics['AUC'].append(auc)
    metrics['Accuracy'].append(accuracy_score(y_true[:, i], y_pred[:, i]))
    metrics['F1'].append(f1_score(y_true[:, i], y_pred[:, i]))
    metrics['Precision'].append(precision_score(y_true[:, i], y_pred[:, i]))
    metrics['Recall'].append(recall_score(y_true[:, i], y_pred[:, i]))

df_metrics = pd.DataFrame(metrics)
df_metrics.loc['mean'] = df_metrics.mean(numeric_only=True)
df_metrics.loc['median'] = df_metrics.median(numeric_only=True)
print(df_metrics)

Epoch 1, Validation AUC: 0.6037
Epoch 2, Validation AUC: 0.6129
Epoch 3, Validation AUC: 0.6389
Epoch 4, Validation AUC: 0.6613
Epoch 5, Validation AUC: 0.6881
Epoch 6, Validation AUC: 0.7117
Epoch 7, Validation AUC: 0.7169
Epoch 8, Validation AUC: 0.7023
Epoch 9, Validation AUC: 0.6975
Epoch 10, Validation AUC: 0.7186
Epoch 11, Validation AUC: 0.7400
Epoch 12, Validation AUC: 0.7343
Epoch 13, Validation AUC: 0.7462
Epoch 14, Validation AUC: 0.7308
Epoch 15, Validation AUC: 0.7279
Epoch 16, Validation AUC: 0.7039
Epoch 17, Validation AUC: 0.7104
Epoch 18, Validation AUC: 0.7285
Epoch 19, Validation AUC: 0.7205
Epoch 20, Validation AUC: 0.7356
Epoch 21, Validation AUC: 0.7422
Epoch 22, Validation AUC: 0.7215
Epoch 23, Validation AUC: 0.7331
Epoch 24, Validation AUC: 0.7204
Epoch 25, Validation AUC: 0.7314
Epoch 26, Validation AUC: 0.7366
Epoch 27, Validation AUC: 0.7280
Epoch 28, Validation AUC: 0.7281
Epoch 29, Validation AUC: 0.7182
Epoch 30, Validation AUC: 0.7400
                Lab

c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Keras (TensorFlow)

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Load data
df = pd.read_csv("tox21.csv")
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]
df = df.dropna(subset=target_cols).reset_index(drop=True)
smiles = df['smiles'].values
labels = df[target_cols].astype(int).values

# Tokenizer
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts(smiles)
sequences = tokenizer.texts_to_sequences(smiles)
X = pad_sequences(sequences, maxlen=120, padding='post')
X_train, X_temp, y_train, y_temp = train_test_split(X, labels, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

# Model
inp = Input(shape=(120,))
x = Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=128)(inp)
x = Bidirectional(LSTM(64))(x)
out = Dense(12, activation='sigmoid')(x)
model = Model(inputs=inp, outputs=out)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=64)

# Prediction
y_score = model.predict(X_test)
y_pred = (y_score >= 0.5).astype(int)

# Metrics
metrics = {
    'Label': [],
    'AUC': [], 'Accuracy': [], 'F1': [], 'Precision': [], 'Recall': []
}

for i, label in enumerate(target_cols):
    try:
        auc = roc_auc_score(y_test[:, i], y_score[:, i])
    except:
        auc = np.nan
    metrics['Label'].append(label)
    metrics['AUC'].append(auc)
    metrics['Accuracy'].append(accuracy_score(y_test[:, i], y_pred[:, i]))
    metrics['F1'].append(f1_score(y_test[:, i], y_pred[:, i]))
    metrics['Precision'].append(precision_score(y_test[:, i], y_pred[:, i]))
    metrics['Recall'].append(recall_score(y_test[:, i], y_pred[:, i]))

df_metrics = pd.DataFrame(metrics)
df_metrics.loc['mean'] = df_metrics.mean(numeric_only=True)
df_metrics.loc['median'] = df_metrics.median(numeric_only=True)
print(df_metrics)


Epoch 1/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 154ms/step - accuracy: 0.0153 - loss: 0.3593 - val_accuracy: 0.0322 - val_loss: 0.1144
Epoch 2/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 135ms/step - accuracy: 0.0625 - loss: 0.1208 - val_accuracy: 0.0322 - val_loss: 0.1137
Epoch 3/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 148ms/step - accuracy: 0.0627 - loss: 0.1113 - val_accuracy: 0.0322 - val_loss: 0.1135
Epoch 4/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 131ms/step - accuracy: 0.0550 - loss: 0.1186 - val_accuracy: 0.0322 - val_loss: 0.1142
Epoch 5/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 138ms/step - accuracy: 0.0626 - loss: 0.1158 - val_accuracy: 0.0386 - val_loss: 0.1124
Epoch 6/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 121ms/step - accuracy: 0.0489 - loss: 0.1221 - val_accuracy: 0.0322 - val_loss: 0.1118
Epoch 7/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 125ms/step - accuracy: 0.0573 - loss: 0.1136 - val_accuracy: 0.0322 - val_loss: 0.1116
Epoch 8/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 146ms/step - accuracy: 0.0604 - loss: 0.1090 - val_accuracy: 0

c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()